# 02 · Primeiros passos com o Weaviate

Os mesmos 25 mil vetores do notebook anterior, agora num banco vetorial open source: persistidos, indexados, com busca semântica, por palavra-chave, híbrida e com filtros.

Pré-requisito: `cd docker && docker compose up -d` (3 nós; o client fala com qualquer um deles).

In [1]:
import time
import weaviate
from weaviate.classes.config import Configure, DataType, Property, Tokenization
from weaviate.classes.query import Filter, MetadataQuery
import utils

client = weaviate.connect_to_local()  # localhost:8080 (REST) + 50051 (gRPC)
print("Weaviate", client.get_meta()["version"])

Weaviate 1.40.0


/Users/dudanogueira/dev/weaviate/lab/claude/latinoware-2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Criar uma collection

Uma **collection** é como uma tabela: tem propriedades (os dados) e um ou mais **vetores**, cada um com seu índice.

Aqui trazemos vetores prontos (`self_provided`). O Weaviate também pode vetorizar sozinho, com módulos para OpenAI, Cohere, Ollama, Hugging Face, Weaviate Embeddings…

In [2]:
client.collections.delete("Wiki")
wiki = client.collections.create(
    "Wiki",
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="text", data_type=DataType.TEXT),
        Property(name="url", data_type=DataType.TEXT, tokenization=Tokenization.FIELD),
    ],
    vector_config=Configure.Vectors.self_provided(),  # índice HNSW, distância cosseno
)

## 2. Importar em lote (batch)

In [3]:
dados = utils.load_wiki(1)

inicio = time.perf_counter()
with wiki.batch.fixed_size(batch_size=500) as batch:
    for linha in dados.itertuples():
        batch.add_object(
            properties={"title": linha.title, "text": linha.text, "url": linha.url},
            vector=linha.vector,
        )
print(f"{len(dados):,} objetos em {time.perf_counter() - inicio:.1f}s | falhas: {len(wiki.batch.failed_objects)}")

utils.wait_for_indexing(client, "Wiki")  # ASYNC_INDEXING=true: o índice termina em segundo plano
print("total na collection:", len(wiki))

25,000 objetos em 4.4s | falhas: 0


total na collection: 25000


## 3. Busca vetorial (semântica)

Pergunta em português, textos em inglês. Vetorizamos a pergunta com o mesmo modelo e pedimos os vizinhos mais próximos.

In [4]:
def mostrar(resposta):
    for o in resposta.objects:
        m = o.metadata
        nota = f"dist={m.distance:.3f}" if m.distance is not None else f"score={m.score:.3f}"
        print(f"{nota}  {o.properties['title']}: {o.properties['text'][:90]}…")


pergunta = "como os vulcões entram em erupção?"
mostrar(wiki.query.near_vector(utils.embed_query(pergunta), limit=3, return_metadata=MetadataQuery(distance=True)))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5996.75it/s]

dist=0.563  Volcanism: Intrusive volcanism is when magma is forced into the rocks that make up the Earth's crust.…
dist=0.582  Intrusion: An intrusion is magma (melted rock) which cools and becomes solid under the Earth's surfac…
dist=0.597  Carbon cycle: The main ways that carbon gets into the carbon cycle are volcanoes, and the burning of fos…


## 4. Busca por palavra-chave (BM25)

BM25 procura **palavras**. Em português, num texto em inglês, ele casa coisas como "os" com… **OS/2**.

In [5]:
mostrar(wiki.query.bm25(pergunta, limit=3, return_metadata=MetadataQuery(score=True)))

score=8.860  OS/2: IBM discontinued its support for OS/2 on 31 December 2006. Since then, it has been updated…
score=7.161  Mac OS X Server 1.0: Mac OS X Server 1.0 is an operating system created by Apple. At first, the OS was called R…
score=6.349  Elis Regina: 1965: O Fino do Fino   – Elis & Zimbo Trio1965: Samba, Eu Canto Assim!1966: Dois na Bossa …


…mas é imbatível para termos exatos: nomes, códigos, siglas, números de série.

In [6]:
mostrar(wiki.query.bm25("Merapi", limit=3, return_metadata=MetadataQuery(score=True)))

score=4.371  Mount Merapi: In 2006, scientists believed that the volcano would erupt soon. An earthquake occurred in …


## 5. Busca híbrida

Combina as duas notas. `alpha=1` é só vetor, `alpha=0` é só BM25. Na prática, comece por volta de 0.5–0.75.

In [7]:
consulta = "Merapi eruption"
mostrar(wiki.query.hybrid(consulta, vector=utils.embed_query(consulta), alpha=0.6, limit=3, return_metadata=MetadataQuery(score=True)))

score=0.633  Mount Merapi: In 2006, scientists believed that the volcano would erupt soon. An earthquake occurred in …
score=0.601  2021 Cumbre Vieja volcanic eruption: The 2021 Cumbre Vieja volcanic eruption was a volcanic eruption of Cumbre Vieja on the isl…
score=0.271  Garibaldi Volcanic Belt: The volcanoes are also the closest to British Columbia's densely populated southwest corne…


## 6. Filtros

Busca vetorial **e** condição estruturada, na mesma consulta. O Weaviate filtra durante a travessia do índice (ACORN), sem perder resultados.

In [8]:
mostrar(
    wiki.query.near_vector(
        utils.embed_query("instrumentos musicais de corda"),
        filters=Filter.by_property("title").like("*guitar*"),
        limit=3,
        return_metadata=MetadataQuery(distance=True),
    )
)

dist=0.631  Electric guitar: The Electric guitar is a type of musical instrument. It was made in 1931 by George Beaucha…
dist=0.696  Guitar: A guitar was described by Dr. Michael Kasha as an instrument that had "a long, fretted nec…
dist=0.724  Bass guitar: The pickups are magnetic devices that sit below the strings. As the strings vibrate, an el…


## 7. Agregações

In [9]:
from weaviate.classes.aggregate import GroupByAggregate

print("objetos:", wiki.aggregate.over_all(total_count=True).total_count)

perto_de_vulcao = wiki.aggregate.near_vector(utils.embed_query("vulcões"), distance=0.65, total_count=True)
print("passagens 'perto' de vulcões:", perto_de_vulcao.total_count)

objetos: 25000
passagens 'perto' de vulcões: 2


Próximo: o que acontece **por baixo** quando criamos (ou deixamos de criar) um schema. ➡️ `03_schema_e_indices_invertidos.ipynb`

In [10]:
client.close()